# Chapter 24: Recommender Systems

## Learning Objectives\n- Understand two-stage recommendation architecture\n- Train matrix factorization with SGD\n- Evaluate top-K ranking quality\n- Handle cold-start users/items

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 24: Recommender Systems

## Why this chapter matters
Recommendation drives major product metrics (watch time, CTR, conversion, retention).
Industry systems combine retrieval, ranking, and post-ranking constraints.

## Learning goals
1. Build user-item collaborative filtering baseline.
2. Understand matrix factorization objective.
3. Add ranking metrics (Precision@K, Recall@K, NDCG@K).
4. Introduce cold-start strategies.

## Two-stage architecture
1. Candidate generation (fast retrieval): ANN search / co-visitation.
2. Ranking model (accurate scoring): gradient boosting or deep model.

## Matrix factorization objective
Minimize:
\[
\sum_{(u,i)\in \Omega}(r_{ui} - p_u^T q_i)^2 + \lambda(\|p_u\|^2 + \|q_i\|^2)
\]

## Offline metrics
- Precision@K
- Recall@K
- MAP@K
- NDCG@K

## Online metrics
- CTR uplift
- conversion uplift
- session depth
- retention

## Assignment
1. Implement MF with SGD from scratch.
2. Evaluate on train/validation split by timestamp.
3. Add popularity fallback for cold users.
4. Write failure analysis by segment.



In [ ]:
from __future__ import annotations

import random
from typing import Dict, List, Tuple


class MatrixFactorization:
    def __init__(self, n_users: int, n_items: int, k: int = 16, lr: float = 0.01, reg: float = 0.01):
        self.k = k
        self.lr = lr
        self.reg = reg
        self.P = [[random.uniform(-0.1, 0.1) for _ in range(k)] for _ in range(n_users)]
        self.Q = [[random.uniform(-0.1, 0.1) for _ in range(k)] for _ in range(n_items)]

    def predict(self, u: int, i: int) -> float:
        return sum(a * b for a, b in zip(self.P[u], self.Q[i]))

    def fit(self, interactions: List[Tuple[int, int, float]], epochs: int = 20) -> None:
        for _ in range(epochs):
            random.shuffle(interactions)
            for u, i, r in interactions:
                pred = self.predict(u, i)
                err = r - pred
                for f in range(self.k):
                    pu = self.P[u][f]
                    qi = self.Q[i][f]
                    self.P[u][f] += self.lr * (err * qi - self.reg * pu)
                    self.Q[i][f] += self.lr * (err * pu - self.reg * qi)


def top_k(model: MatrixFactorization, user: int, n_items: int, seen: Dict[int, set], k: int = 5) -> List[int]:
    scored = []
    blocked = seen.get(user, set())
    for item in range(n_items):
        if item in blocked:
            continue
        scored.append((model.predict(user, item), item))
    scored.sort(reverse=True)
    return [item for _, item in scored[:k]]


if __name__ == "__main__":
    # (user, item, rating/implicit score)
    data = [
        (0, 0, 1.0), (0, 1, 1.0), (1, 1, 1.0), (1, 2, 1.0),
        (2, 2, 1.0), (2, 3, 1.0), (3, 3, 1.0), (3, 4, 1.0),
    ]
    seen = {0: {0, 1}, 1: {1, 2}, 2: {2, 3}, 3: {3, 4}}

    mf = MatrixFactorization(n_users=4, n_items=5, k=8, lr=0.05, reg=0.01)
    mf.fit(data, epochs=80)

    for u in range(4):
        print(f"User {u} recommendations:", top_k(mf, u, n_items=5, seen=seen, k=2))


## Checkpoint\n\n1. You can explain candidate generation vs ranking.\n2. You can train latent factor vectors with SGD.\n3. You can define at least two top-K metrics.

## Guided Exercise\nImplement Precision@K for held-out interactions and compare users.

In [ ]:
# TODO (Student Exercise)
def precision_at_k(pred_items, true_items, k):
    hit = len(set(pred_items[:k]) & set(true_items))
    return hit / k

print(precision_at_k([1,2,3], [2,7], 3))

## Exercise Solution

In [ ]:
def recall_at_k(pred_items, true_items, k):
    hit = len(set(pred_items[:k]) & set(true_items))
    return hit / max(1, len(true_items))

print('recall@3', recall_at_k([1,2,3], [2,7], 3))

## Quick Quiz\n\n**Q1.** Why do recommender systems use a two-stage setup?\n\n**Answer:** To balance retrieval speed with ranking accuracy.\n\n**Q2.** What is cold-start?\n\n**Answer:** Insufficient interaction history for new users/items.\n\n**Q3.** One cold-start fallback?\n\n**Answer:** Popularity-based recommendations by segment.\n

## Homework\nAdd user and item bias terms to the MF objective and measure gain.